### Step 1: Install required libraries

In [1]:
!pip install datasets pandas pymupdf pageindex groq python-dotenv ipykernel openai


  Using cached datasets-4.8.4-py3-none-any.whl.metadata (19 kB)
  Using cached pandas-3.0.2-cp312-cp312-win_amd64.whl.metadata (19 kB)
  Using cached pymupdf-1.27.2.2-cp310-abi3-win_amd64.whl.metadata (3.4 kB)
  Using cached pageindex-0.2.8-py3-none-any.whl.metadata (628 bytes)
  Using cached groq-1.2.0-py3-none-any.whl.metadata (16 kB)
  Using cached python_dotenv-1.2.2-py3-none-any.whl.metadata (27 kB)
  Using cached openai-2.32.0-py3-none-any.whl.metadata (31 kB)
  Using cached filelock-3.29.0-py3-none-any.whl.metadata (2.0 kB)
  Using cached numpy-2.4.4-cp312-cp312-win_amd64.whl.metadata (6.6 kB)
  Using cached pyarrow-23.0.1-cp312-cp312-win_amd64.whl.metadata (3.1 kB)
  Using cached dill-0.4.1-py3-none-any.whl.metadata (10 kB)
  Using cached requests-2.33.1-py3-none-any.whl.metadata (4.8 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached tqdm-4.67.3-py3-none-any.whl.metadata (57 kB)
  Using cached xxhash-3.6.0-cp312-cp312-win_amd64.whl.metadata (13 k

In [4]:
!pip install -U ipywidgets jupyter


  Using cached ipywidgets-8.1.8-py3-none-any.whl.metadata (2.4 kB)
  Using cached jupyter-1.1.1-py2.py3-none-any.whl.metadata (2.0 kB)
  Using cached widgetsnbextension-4.0.15-py3-none-any.whl.metadata (1.6 kB)
  Using cached jupyterlab_widgets-3.0.16-py3-none-any.whl.metadata (20 kB)
  Using cached jupyter_console-6.6.3-py3-none-any.whl.metadata (5.8 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached jupyter_server-2.17.0-py3-none-any.whl.metadata (8.5 kB)
  Using cached jupyterlab_server-2.28.0-py3-none-any.whl.metadata (5.9 kB)
  Using cached notebook_shim-0.2.4-py3-none-any.whl.metadata (4.0 kB)
  Using cached argon2_cffi-25.1.0-py3-none-any.whl.metadata (4.1 kB)
  Using cached jupyter_events-0.12.0-py3-none-any.whl.metadata (5.8 kB)
  Using cached nbformat-5.10.4-py3-none-any.whl.metadata (3.6 kB)
  Using cached terminado-0.18.1-py3-none-any.whl.metadata (5.8 kB)
  Using cached websocket_client-1.9.0-py3-none-any.whl.metadata (8.3 kB)
  Using cached 

In [5]:
import datasets
import pandas as pd
import fitz          # PyMuPDF
import pageindex
import groq
import dotenv
import openai

print("All imports successful")

import ipywidgets
print("ipywidgets installed")


All imports successful
ipywidgets installed


### Step 2: Downloand dataset

In [6]:
from datasets import load_dataset
import pandas as pd

# Download dataset
dataset = load_dataset("RahulS3/siddha_vaithiyam_question_answering_chatbot")

print(dataset)

# Convert train split to DataFrame
df = pd.DataFrame(dataset["train"])

# Save locally
df.to_csv("siddha_dataset.csv", index=False, encoding="utf-8")

print("Saved as siddha_dataset.csv")

# Inspect columns and first rows
print(df.columns)
print(df.head())


DatasetDict({
    train: Dataset({
        features: ['Questions', 'Answers'],
        num_rows: 1084
    })
})
Saved as siddha_dataset.csv
Index(['Questions', 'Answers'], dtype='str')
             Questions                                            Answers
0         குடியை மறக்க  வில்வ இலை, மிளகு, கொத்தமல்லி விதை இந்த மூன்றைய...
1             செல்போன்  செல்போன் அடிக்கடி உபயோகித்தால் மூளை மந்தம், கா...
2        முகம் பளபளக்க  நாட்டு வாழைப்பழம் நன்றாக பழுத்தது. ஆலிவ் ஆயில்...
3  முகச்சுருக்கம் மறைய             முட்டைகோஸ் சாறை முகத்தில் தடவி வரலாம்.
4   உடல் மினுமினுப்பாக  இரவில் படுக்கப் போகும் முன் தேன், குங்குமப்பூ....


### Step 3:Clean the dataset

In [7]:
# Check original columns first
print(df.columns)

# Rename columns
df = df.rename(columns={
    "Questions": "question",
    "Answers": "answer"
})

# Keep only required columns
df = df[["question", "answer"]]

# Remove nulls
df = df.dropna()

# Convert to string and strip spaces
df["question"] = df["question"].astype(str).str.strip()
df["answer"] = df["answer"].astype(str).str.strip()

# Remove empty rows
df = df[(df["question"] != "") & (df["answer"] != "")]

# Optional: remove duplicate question-answer pairs
df = df.drop_duplicates(subset=["question", "answer"]).reset_index(drop=True)

# Add serial number
df.insert(0, "s.no", range(1, len(df) + 1))

# Save clean CSV
df.to_csv("siddha_clean.csv", index=False, encoding="utf-8")

print("Clean CSV saved!")
print("Total rows:", len(df))

df.head()


Index(['Questions', 'Answers'], dtype='str')
Clean CSV saved!
Total rows: 1082


,s.no,question,answer
0,1,குடியை மறக்க,"வில்வ இலை, மிளகு, கொத்தமல்லி விதை இந்த மூன்றைய..."
1,2,செல்போன்,"செல்போன் அடிக்கடி உபயோகித்தால் மூளை மந்தம், கா..."
2,3,முகம் பளபளக்க,நாட்டு வாழைப்பழம் நன்றாக பழுத்தது. ஆலிவ் ஆயில்...
3,4,முகச்சுருக்கம் மறைய,முட்டைகோஸ் சாறை முகத்தில் தடவி வரலாம்.
4,5,உடல் மினுமினுப்பாக,"இரவில் படுக்கப் போகும் முன் தேன், குங்குமப்பூ...."


In [8]:
import pandas as pd

df = pd.read_csv("siddha_clean.csv")

print(df.head())
print("Total rows:", len(df))

clean_data = []

for _, row in df.iterrows():
    q = str(row["question"]).strip()
    a = str(row["answer"]).strip()

    q = " ".join(q.split())
    a = " ".join(a.split())

    clean_data.append({
        "id": int(row["s.no"]),
        "question": q,
        "answer": a
    })

print(clean_data[:3])

def chunk_data(data, size=50):
    for i in range(0, len(data), size):
        yield data[i:i+size]

batch = list(chunk_data(clean_data, 25))[0]

print(len(batch))
print(batch[0])
print(batch[1])


   s.no             question  \
0     1         குடியை மறக்க   
1     2             செல்போன்   
2     3        முகம் பளபளக்க   
3     4  முகச்சுருக்கம் மறைய   
4     5   உடல் மினுமினுப்பாக   

                                              answer  
0  வில்வ இலை, மிளகு, கொத்தமல்லி விதை இந்த மூன்றைய...  
1  செல்போன் அடிக்கடி உபயோகித்தால் மூளை மந்தம், கா...  
2  நாட்டு வாழைப்பழம் நன்றாக பழுத்தது. ஆலிவ் ஆயில்...  
3             முட்டைகோஸ் சாறை முகத்தில் தடவி வரலாம்.  
4  இரவில் படுக்கப் போகும் முன் தேன், குங்குமப்பூ....  
Total rows: 1082
[{'id': 1, 'question': 'குடியை மறக்க', 'answer': 'வில்வ இலை, மிளகு, கொத்தமல்லி விதை இந்த மூன்றையும் 300 மி. தண்ணீர் விட்டு கொதிக்க வைத்து இறக்கவும். வாரம் ஒரு முறை மாதம் 4 முறை கொடுக்கவும். பலன் கிடைக்கும்'}, {'id': 2, 'question': 'செல்போன்', 'answer': 'செல்போன் அடிக்கடி உபயோகித்தால் மூளை மந்தம், காது கோளாறு போன்றவை ஏற்படும்'}, {'id': 3, 'question': 'முகம் பளபளக்க', 'answer': 'நாட்டு வாழைப்பழம் நன்றாக பழுத்தது. ஆலிவ் ஆயில் சேர்த்து பிசைந்து முகத்தில் தடவி 

### Step 4: LLM core

#### Step 4.1 classify lesson names

In [12]:
import os
import json
import time
import pandas as pd
from dotenv import load_dotenv
from groq import Groq

# Load API key
load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")

if not GROQ_API_KEY:
    raise ValueError("GROQ_API_KEY not found. Add it to your .env file.")

groq_client = Groq(api_key=GROQ_API_KEY)

# Load cleaned dataset
df = pd.read_csv("siddha_clean.csv")

print("Total rows:", len(df))
print(df.head())


Total rows: 1082
   s.no             question  \
0     1         குடியை மறக்க   
1     2             செல்போன்   
2     3        முகம் பளபளக்க   
3     4  முகச்சுருக்கம் மறைய   
4     5   உடல் மினுமினுப்பாக   

                                              answer  
0  வில்வ இலை, மிளகு, கொத்தமல்லி விதை இந்த மூன்றைய...  
1  செல்போன் அடிக்கடி உபயோகித்தால் மூளை மந்தம், கா...  
2  நாட்டு வாழைப்பழம் நன்றாக பழுத்தது. ஆலிவ் ஆயில்...  
3             முட்டைகோஸ் சாறை முகத்தில் தடவி வரலாம்.  
4  இரவில் படுக்கப் போகும் முன் தேன், குங்குமப்பூ....  


In [13]:
# Convert CSV rows into clean dict format
clean_data = []

for _, row in df.iterrows():
    q = str(row["question"]).strip()
    a = str(row["answer"]).strip()

    q = " ".join(q.split())
    a = " ".join(a.split())

    clean_data.append({
        "id": int(row["s.no"]),
        "question": q,
        "answer": a
    })

print("Prepared rows:", len(clean_data))
print(clean_data[:2])


Prepared rows: 1082
[{'id': 1, 'question': 'குடியை மறக்க', 'answer': 'வில்வ இலை, மிளகு, கொத்தமல்லி விதை இந்த மூன்றையும் 300 மி. தண்ணீர் விட்டு கொதிக்க வைத்து இறக்கவும். வாரம் ஒரு முறை மாதம் 4 முறை கொடுக்கவும். பலன் கிடைக்கும்'}, {'id': 2, 'question': 'செல்போன்', 'answer': 'செல்போன் அடிக்கடி உபயோகித்தால் மூளை மந்தம், காது கோளாறு போன்றவை ஏற்படும்'}]


In [14]:
ALLOWED_LESSONS = [
    "Skin Care and Beauty",
    "Hair Care",
    "Women Health",
    "Pregnancy and Postpartum",
    "Child Care",
    "Male Health",
    "Digestive Health",
    "Respiratory Health",
    "Fever and Infections",
    "ENT, Mouth and Dental",
    "Eye Care",
    "Pain, Joints and Nerves",
    "Wounds, Burns and Injuries",
    "Heart, Blood and Circulation",
    "Liver, Bile and Jaundice",
    "Kidney, Urinary and Diabetes",
    "Mental Health, Sleep and Memory",
    "Poison, Bites and Stings",
    "General Wellness and Body Strength"
]


In [15]:
def chunk_data(data, size=20):
    for i in range(0, len(data), size):
        yield data[i:i + size]


def build_lesson_prompt(batch):
    compact_batch = [
        {
            "id": int(x["id"]),
            "question": x["question"]
        }
        for x in batch
    ]

    return f"""
Return ONLY a valid JSON array.
No explanation.
No markdown.
No code fences.

Classify each Siddha medicine question into exactly ONE lesson.

Allowed lessons:
{ALLOWED_LESSONS}

Return format:
[
  {{
    "id": 1,
    "lesson_en": "Skin Care and Beauty"
  }}
]

Rules:
- Use only the exact lesson names from the allowed list.
- Do not create new lesson names.
- Include every input id exactly once.
- Do not include answer text.
- If the question is about face, skin glow, pimples, itching, scabies, rashes, scars, or complexion, choose "Skin Care and Beauty".
- If the question is about hair growth, dandruff, lice, baldness, grey hair, or hair fall, choose "Hair Care".
- If the question is about menstruation, uterus, white discharge, female infertility, breast health, or lactation, choose "Women Health".
- If the question is about pregnancy, fetus, delivery, miscarriage, or postpartum recovery, choose "Pregnancy and Postpartum".
- If the question is specifically about babies or children, choose "Child Care".
- If the question is about male fertility, semen, virility, male reproductive organs, or male sexual strength, choose "Male Health".
- If the question is about stomach, digestion, constipation, diarrhea, piles, gas, vomiting, appetite, hiccups, mouth ulcer connected with digestion, or intestinal worms, choose "Digestive Health".
- If the question is about cough, cold, phlegm, asthma, breathing, lungs, chest congestion, or respiratory allergy, choose "Respiratory Health".
- If the question is about fever, typhoid, malaria, smallpox, infection, or infectious disease, choose "Fever and Infections".
- If the question is about ear, nose, throat, mouth, tongue, lips, teeth, gums, voice, or bad breath, choose "ENT, Mouth and Dental".
- If the question is about eyes or eyesight, choose "Eye Care".
- If the question is about body pain, joint pain, nerve pain, back pain, neck pain, paralysis, sprain, swelling, leg/hand pain, or vatha conditions, choose "Pain, Joints and Nerves".
- If the question is about cuts, wounds, burns, ulcers, injuries, bleeding wounds, bed sores, nail wounds, or corns, choose "Wounds, Burns and Injuries".
- If the question is about heart, blood pressure, blood purification, anemia, circulation, chest pain related to heart, or blood disorders, choose "Heart, Blood and Circulation".
- If the question is about liver, bile, pitham, jaundice, spleen, or gall bladder, choose "Liver, Bile and Jaundice".
- If the question is about urine, kidney, urinary stones, diabetes, excessive urination, or urinary burning, choose "Kidney, Urinary and Diabetes".
- If the question is about mind, brain, memory, sleep, dizziness, mental illness, or hysteria, choose "Mental Health, Sleep and Memory".
- If the question is about snake bite, dog bite, scorpion sting, insect bite, poison, venom, or toxic bites, choose "Poison, Bites and Stings".
- If none of the above fit, choose "General Wellness and Body Strength".

Data:
{json.dumps(compact_batch, ensure_ascii=False)}
"""


In [16]:
all_assignments = []

batches = list(chunk_data(clean_data, size=20))

print("Total batches:", len(batches))

for batch_no, batch in enumerate(batches, start=1):
    print(f"Processing batch {batch_no}/{len(batches)}")

    response = groq_client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {
                "role": "user",
                "content": build_lesson_prompt(batch)
            }
        ],
        temperature=0,
        max_completion_tokens=1500
    )

    raw_output = response.choices[0].message.content.strip()

    try:
        assignments = json.loads(raw_output)
    except json.JSONDecodeError:
        print("JSON parse failed in batch:", batch_no)
        print(raw_output)
        raise

    input_ids = {item["id"] for item in batch}
    output_ids = {item["id"] for item in assignments}

    missing_ids = input_ids - output_ids
    extra_ids = output_ids - input_ids

    if missing_ids:
        raise ValueError(f"Missing ids in batch {batch_no}: {missing_ids}")

    if extra_ids:
        raise ValueError(f"Extra ids in batch {batch_no}: {extra_ids}")

    for item in assignments:
        if item["lesson_en"] not in ALLOWED_LESSONS:
            raise ValueError(f"Invalid lesson in batch {batch_no}: {item}")

    all_assignments.extend(assignments)

    time.sleep(1)

print("Classification completed")
print("Total classified:", len(all_assignments))


Total batches: 55
Processing batch 1/55
Processing batch 2/55
Processing batch 3/55
Processing batch 4/55
Processing batch 5/55
Processing batch 6/55
Processing batch 7/55
Processing batch 8/55
Processing batch 9/55
Processing batch 10/55
Processing batch 11/55
Processing batch 12/55
Processing batch 13/55
Processing batch 14/55
Processing batch 15/55
Processing batch 16/55
Processing batch 17/55
Processing batch 18/55
Processing batch 19/55
Processing batch 20/55
Processing batch 21/55
Processing batch 22/55
Processing batch 23/55
Processing batch 24/55
Processing batch 25/55
Processing batch 26/55
Processing batch 27/55
Processing batch 28/55
Processing batch 29/55
Processing batch 30/55
Processing batch 31/55
Processing batch 32/55
Processing batch 33/55
Processing batch 34/55
Processing batch 35/55
Processing batch 36/55
Processing batch 37/55
Processing batch 38/55
Processing batch 39/55
Processing batch 40/55
Processing batch 41/55


RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01kpbtq09peg1sj17gws7fmskt` service tier `on_demand` on tokens per day (TPD): Limit 100000, Used 99113, Requested 2231. Please try again in 19m21.216s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}

In [17]:
import pandas as pd

# Save whatever was completed before rate limit
partial_lesson_df = pd.DataFrame(all_assignments)

print("Saved classifications:", len(partial_lesson_df))
print(partial_lesson_df.head())
print(partial_lesson_df["lesson_en"].value_counts())

partial_lesson_df.to_csv(
    "siddha_lesson_assignments_partial.csv",
    index=False,
    encoding="utf-8"
)

print("Saved as siddha_lesson_assignments_partial.csv")


Saved classifications: 800
   id                           lesson_en
0   1  General Wellness and Body Strength
1   2  General Wellness and Body Strength
2   3                Skin Care and Beauty
3   4                Skin Care and Beauty
4   5  General Wellness and Body Strength
lesson_en
Digestive Health                      115
Pain, Joints and Nerves                78
Respiratory Health                     72
Wounds, Burns and Injuries             71
Skin Care and Beauty                   65
ENT, Mouth and Dental                  62
Liver, Bile and Jaundice               54
Kidney, Urinary and Diabetes           51
Heart, Blood and Circulation           40
Women Health                           38
General Wellness and Body Strength     35
Eye Care                               28
Mental Health, Sleep and Memory        23
Fever and Infections                   23
Hair Care                              19
Male Health                            12
Child Care                             

In [18]:
partial_final_df = df.merge(
    partial_lesson_df,
    left_on="s.no",
    right_on="id",
    how="left"
)

partial_final_df = partial_final_df[["s.no", "question", "answer", "lesson_en"]]

print("Total rows:", len(partial_final_df))
print("Classified rows:", partial_final_df["lesson_en"].notna().sum())
print("Missing rows:", partial_final_df["lesson_en"].isna().sum())

partial_final_df.to_csv(
    "siddha_with_lessons_partial.csv",
    index=False,
    encoding="utf-8"
)

print("Saved as siddha_with_lessons_partial.csv")


Total rows: 1082
Classified rows: 800
Missing rows: 282
Saved as siddha_with_lessons_partial.csv


In [20]:
import pandas as pd

# Load original clean dataset
df = pd.read_csv("siddha_clean.csv")

# Load whatever lesson classifications were saved
lesson_df = pd.read_csv("siddha_lesson_assignments_partial.csv")

print("Original rows:", len(df))
print("Classified rows before cleaning:", len(lesson_df))

lesson_df.head()


Original rows: 1082
Classified rows before cleaning: 800


,id,lesson_en
0,1,General Wellness and Body Strength
1,2,General Wellness and Body Strength
2,3,Skin Care and Beauty
3,4,Skin Care and Beauty
4,5,General Wellness and Body Strength


In [21]:
final_df = df.merge(
    partial_lesson_df,
    left_on="s.no",
    right_on="id",
    how="left"
)

final_df = final_df[["s.no", "question", "answer", "lesson_en"]]

print("Total rows:", len(final_df))
print("Rows with lesson:", final_df["lesson_en"].notna().sum())
print("Rows without lesson:", final_df["lesson_en"].isna().sum())


Total rows: 1082
Rows with lesson: 800
Rows without lesson: 282


In [22]:
usable_df = final_df.dropna(subset=["lesson_en"]).copy()

usable_df["question"] = usable_df["question"].astype(str).str.strip()
usable_df["answer"] = usable_df["answer"].astype(str).str.strip()
usable_df["lesson_en"] = usable_df["lesson_en"].astype(str).str.strip()

usable_df = usable_df[
    (usable_df["question"] != "") &
    (usable_df["answer"] != "") &
    (usable_df["lesson_en"] != "")
]

usable_df = usable_df.sort_values(["lesson_en", "s.no"]).reset_index(drop=True)

print("Usable rows:", len(usable_df))
print(usable_df["lesson_en"].value_counts())

usable_df.to_csv(
    "siddha_with_lessons_clean.csv",
    index=False,
    encoding="utf-8"
)

print("Saved as siddha_with_lessons_clean.csv")


Usable rows: 800
lesson_en
Digestive Health                      115
Pain, Joints and Nerves                78
Respiratory Health                     72
Wounds, Burns and Injuries             71
Skin Care and Beauty                   65
ENT, Mouth and Dental                  62
Liver, Bile and Jaundice               54
Kidney, Urinary and Diabetes           51
Heart, Blood and Circulation           40
Women Health                           38
General Wellness and Body Strength     35
Eye Care                               28
Fever and Infections                   23
Mental Health, Sleep and Memory        23
Hair Care                              19
Male Health                            12
Child Care                              8
Pregnancy and Postpartum                5
Poison, Bites and Stings                1
Name: count, dtype: int64
Saved as siddha_with_lessons_clean.csv


### Step 5:Merge and make it as Markdown

In [28]:
import re
from pathlib import Path
import pandas as pd

df = pd.read_csv("siddha_with_lessons_clean.csv")

print("Rows loaded:", len(df))
print(df["lesson_en"].value_counts())


Rows loaded: 800
lesson_en
Digestive Health                      115
Pain, Joints and Nerves                78
Respiratory Health                     72
Wounds, Burns and Injuries             71
Skin Care and Beauty                   65
ENT, Mouth and Dental                  62
Liver, Bile and Jaundice               54
Kidney, Urinary and Diabetes           51
Heart, Blood and Circulation           40
Women Health                           38
General Wellness and Body Strength     35
Eye Care                               28
Fever and Infections                   23
Mental Health, Sleep and Memory        23
Hair Care                              19
Male Health                            12
Child Care                              8
Pregnancy and Postpartum                5
Poison, Bites and Stings                1
Name: count, dtype: int64


In [29]:
LESSON_ORDER = [
    "Skin Care and Beauty",
    "Hair Care",
    "Women Health",
    "Pregnancy and Postpartum",
    "Child Care",
    "Male Health",
    "Digestive Health",
    "Respiratory Health",
    "Fever and Infections",
    "ENT, Mouth and Dental",
    "Eye Care",
    "Pain, Joints and Nerves",
    "Wounds, Burns and Injuries",
    "Heart, Blood and Circulation",
    "Liver, Bile and Jaundice",
    "Kidney, Urinary and Diabetes",
    "Mental Health, Sleep and Memory",
    "Poison, Bites and Stings",
    "General Wellness and Body Strength"
]

def clean_text(text):
    text = str(text).strip()
    text = " ".join(text.split())
    return text


In [30]:
lines = []

lines.append("# Siddha Vaithiyam Knowledge Base")
lines.append("")
lines.append("This knowledge base contains Siddha medicine question-answer records grouped by lesson.")
lines.append("")
lines.append("Important: This content is for educational use only and should not replace professional medical advice.")
lines.append("")
lines.append("---")
lines.append("")

# Index section
lines.append("# Index")
lines.append("")

lesson_summaries = []

for lesson_number, lesson_name in enumerate(LESSON_ORDER, start=1):
    lesson_df = df[df["lesson_en"] == lesson_name].copy()

    if lesson_df.empty:
        continue

    lesson_summaries.append({
        "lesson_number": lesson_number,
        "lesson_name": lesson_name,
        "count": len(lesson_df)
    })

for item in lesson_summaries:
    lines.append(
        f'{item["lesson_number"]}. {item["lesson_name"]} - {item["count"]} Q&A items'
    )

lines.append("")
lines.append("## Summary")
lines.append("")
lines.append(f"- Total lessons: {len(lesson_summaries)}")
lines.append(f"- Total Q&A items: {len(df)}")
lines.append("")
lines.append("---")
lines.append("")


In [31]:
# Lesson sections
for item in lesson_summaries:
    lesson_number = item["lesson_number"]
    lesson_name = item["lesson_name"]

    lesson_df = df[df["lesson_en"] == lesson_name].copy()
    lesson_df = lesson_df.sort_values("s.no")

    lines.append(f"# Lesson {lesson_number}: {lesson_name}")
    lines.append("")
    lines.append(f"Total Q&A items: {len(lesson_df)}")
    lines.append("")
    lines.append("---")
    lines.append("")

    for _, row in lesson_df.iterrows():
        item_id = int(row["s.no"])
        question = clean_text(row["question"])
        answer = clean_text(row["answer"])

        lines.append(f"## {item_id}. {question}")
        lines.append("")
        lines.append("**Answer:**")
        lines.append("")
        lines.append(answer)
        lines.append("")
        lines.append("---")
        lines.append("")

output_path = Path("siddha_knowledge_base.md")
output_path.write_text("\n".join(lines), encoding="utf-8")

print("Saved:", output_path)
print("Total lessons:", len(lesson_summaries))
print("Total Q&A items:", len(df))


Saved: siddha_knowledge_base.md
Total lessons: 19
Total Q&A items: 800


In [32]:
print(Path("siddha_knowledge_base.md").read_text(encoding="utf-8")[:3000])


# Siddha Vaithiyam Knowledge Base

This knowledge base contains Siddha medicine question-answer records grouped by lesson.

Important: This content is for educational use only and should not replace professional medical advice.

---

# Index

1. Skin Care and Beauty - 65 Q&A items
2. Hair Care - 19 Q&A items
3. Women Health - 38 Q&A items
4. Pregnancy and Postpartum - 5 Q&A items
5. Child Care - 8 Q&A items
6. Male Health - 12 Q&A items
7. Digestive Health - 115 Q&A items
8. Respiratory Health - 72 Q&A items
9. Fever and Infections - 23 Q&A items
10. ENT, Mouth and Dental - 62 Q&A items
11. Eye Care - 28 Q&A items
12. Pain, Joints and Nerves - 78 Q&A items
13. Wounds, Burns and Injuries - 71 Q&A items
14. Heart, Blood and Circulation - 40 Q&A items
15. Liver, Bile and Jaundice - 54 Q&A items
16. Kidney, Urinary and Diabetes - 51 Q&A items
17. Mental Health, Sleep and Memory - 23 Q&A items
18. Poison, Bites and Stings - 1 Q&A items
19. General Wellness and Body Strength - 35 Q&A items



## Part - 2

In [34]:
import json
import pandas as pd
from pathlib import Path

df = pd.read_csv("siddha_with_lessons_clean.csv")

LESSON_ORDER = [
    "Skin Care and Beauty",
    "Hair Care",
    "Women Health",
    "Pregnancy and Postpartum",
    "Child Care",
    "Male Health",
    "Digestive Health",
    "Respiratory Health",
    "Fever and Infections",
    "ENT, Mouth and Dental",
    "Eye Care",
    "Pain, Joints and Nerves",
    "Wounds, Burns and Injuries",
    "Heart, Blood and Circulation",
    "Liver, Bile and Jaundice",
    "Kidney, Urinary and Diabetes",
    "Mental Health, Sleep and Memory",
    "Poison, Bites and Stings",
    "General Wellness and Body Strength"
]

def clean_text(text):
    text = str(text).strip()
    text = " ".join(text.split())
    return text

tree = {
    "title": "Siddha Vaithiyam Knowledge Base",
    "description": "Siddha medicine question-answer records grouped by lesson.",
    "total_items": int(len(df)),
    "lessons": []
}

for lesson_id, lesson_name in enumerate(LESSON_ORDER, start=1):
    lesson_df = df[df["lesson_en"] == lesson_name].copy()

    if lesson_df.empty:
        continue

    lesson_df = lesson_df.sort_values("s.no")

    lesson = {
        "lesson_id": lesson_id,
        "lesson_name": lesson_name,
        "items_count": int(len(lesson_df)),
        "items": []
    }

    for _, row in lesson_df.iterrows():
        lesson["items"].append({
            "id": int(row["s.no"]),
            "question": clean_text(row["question"]),
            "answer": clean_text(row["answer"])
        })

    tree["lessons"].append(lesson)

Path("siddha_tree.json").write_text(
    json.dumps(tree, ensure_ascii=False, indent=2),
    encoding="utf-8"
)

print("Saved siddha_tree.json")
print("Total lessons:", len(tree["lessons"]))
print("Total items:", tree["total_items"])


Saved siddha_tree.json
Total lessons: 19
Total items: 800


In [35]:
import json
from pathlib import Path

TREE_PATH = Path("siddha_tree.json")

tree = json.loads(TREE_PATH.read_text(encoding="utf-8"))

print(f"Top-level lessons: {len(tree['lessons'])}")
print(f"Total items: {tree['total_items']}")
print()
print("First lesson preview:")
print(json.dumps(tree["lessons"][0], ensure_ascii=False, indent=2)[:1000])


Top-level lessons: 19
Total items: 800

First lesson preview:
{
  "lesson_id": 1,
  "lesson_name": "Skin Care and Beauty",
  "items_count": 65,
  "items": [
    {
      "id": 3,
      "question": "முகம் பளபளக்க",
      "answer": "நாட்டு வாழைப்பழம் நன்றாக பழுத்தது. ஆலிவ் ஆயில் சேர்த்து பிசைந்து முகத்தில் தடவி 1 மணி நேரம் கழித்து முகம் கழுவி வரலாம்"
    },
    {
      "id": 4,
      "question": "முகச்சுருக்கம் மறைய",
      "answer": "முட்டைகோஸ் சாறை முகத்தில் தடவி வரலாம்."
    },
    {
      "id": 6,
      "question": "உடல் நிறம் பளபளக்க",
      "answer": "அவரி இலையை சுத்தம் செய்து நன்கு உலர்த்தி தூளாக்கி தினமும் 5 கிராம் காலை உணவிற்க பின் சாப்பிடவும்"
    },
    {
      "id": 8,
      "question": "முக வசீகரம்",
      "answer": "சந்தன கட்டை எலுமிச்சை சாறில் உரைத்து பூசலாம்"
    },
    {
      "id": 9,
      "question": "முகம் பிரகாசமடைய",
      "answer": "கானாவாழை, மாவிலை சமஅளவு எடுத்து காய்ச்சி வடிகட்டி அதை முகத்தில் தடவி காயவிட்டு அரை மணிநேரம் கழித்து கழுவவும்."
    },
    {
      "id"

In [36]:
def print_siddha_tree(tree, max_items_per_lesson=5):
    print(tree["title"])
    print("Total lessons:", len(tree["lessons"]))
    print("Total items:", tree["total_items"])
    print()

    for lesson in tree["lessons"]:
        lesson_id = lesson["lesson_id"]
        lesson_name = lesson["lesson_name"]
        items_count = lesson["items_count"]

        print(f"Lesson {lesson_id}: {lesson_name} ({items_count} items)")

        for item in lesson["items"][:max_items_per_lesson]:
            print(f"  - [{item['id']}] {item['question']}")

        remaining = items_count - max_items_per_lesson

        if remaining > 0:
            print(f"  ... {remaining} more")

        print()

print_siddha_tree(tree)


Siddha Vaithiyam Knowledge Base
Total lessons: 19
Total items: 800

Lesson 1: Skin Care and Beauty (65 items)
  - [3] முகம் பளபளக்க
  - [4] முகச்சுருக்கம் மறைய
  - [6] உடல் நிறம் பளபளக்க
  - [8] முக வசீகரம்
  - [9] முகம் பிரகாசமடைய
  ... 60 more

Lesson 2: Hair Care (19 items)
  - [90] வேர் குரு
  - [93] சிரட்டை தைலம்
  - [517] இளவயது நரை மாற
  - [518] இளம் நரை மறைய
  - [519] இளநரை தீர, உடல் பலம் பெற
  ... 14 more

Lesson 3: Women Health (38 items)
  - [21] குழந்தையின்மை நீங்க
  - [22] கர்ப்பபை புழு நீங்க
  - [24] பிள்ளைப் பேறு உண்டாக
  - [25] கர்ப்பபை நோய்கள் தீர
  - [26] பெரும்பாடு தீர, கர்ப்பபை பலப்பட
  ... 33 more

Lesson 4: Pregnancy and Postpartum (5 items)
  - [55] கருவுற்ற தாய்மார்கள்
  - [56] கர்ப்பாயாச கோளாறு நீங்க
  - [57] கருச்சிதைவு ஏற்படாமல் இருக்க
  - [700] குழந்தை பிறப்பதற்கு முன்னும், பிறந்த பின்னும்
  - [705] கர்ப்ப காலத்தில் வாந்தி மட்டுபட

Lesson 5: Child Care (8 items)
  - [23] குழந்தை சிவப்பாக பிறக்க
  - [41] குழந்தைகளுக்கு
  - [42] போலியோ சொட்டு மருந்து
  - [44] 

In [37]:
def count_retrievable_items(tree):
    total_lessons = len(tree["lessons"])
    total_items = sum(lesson["items_count"] for lesson in tree["lessons"])

    return total_lessons, total_items

total_lessons, total_items = count_retrievable_items(tree)

print(f"Total lessons: {total_lessons}")
print(f"Total retrievable Q&A items: {total_items}")
print("Each Q&A item = one retrievable chunk.")


Total lessons: 19
Total retrievable Q&A items: 800
Each Q&A item = one retrievable chunk.


### Step 1: Select Relevant Lessons

In [54]:
import os
import json
import re
from pathlib import Path
from dotenv import load_dotenv
from groq import Groq

load_dotenv()

GRO_API_KEY = os.getenv("GRO_API_KEY")

if not GRO_API_KEY:
    raise ValueError("GROQ_API_KEY missing in .env")

groq_client = Groq(api_key=GRO_API_KEY)

tree = json.loads(Path("siddha_tree.json").read_text(encoding="utf-8"))

print("Tree loaded")
print("Lessons:", len(tree["lessons"]))
print("Items:", tree["total_items"])


Tree loaded
Lessons: 19
Items: 800


In [63]:
def compact_json(data):
    return json.dumps(data, ensure_ascii=False, separators=(",", ":"))


def parse_json_response(text):
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        match = re.search(r"\{.*\}", text, re.DOTALL)
        if match:
            return json.loads(match.group())
        return {}


In [64]:
def groq_json(prompt, model="llama-3.3-70b-versatile", max_tokens=300):
    response = groq_client.chat.completions.create(
        model=model,
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0,
        max_completion_tokens=max_tokens
    )

    content = response.choices[0].message.content.strip()
    return parse_json_response(content)


In [65]:
def llm_select_lessons(query, tree, model="llama-3.3-70b-versatile"):
    lesson_index = []

    for lesson in tree["lessons"]:
        sample_questions = [
            item["question"]
            for item in lesson["items"][:8]
        ]

        lesson_index.append({
            "lesson_id": lesson["lesson_id"],
            "lesson_name": lesson["lesson_name"],
            "items_count": lesson["items_count"],
            "sample_questions": sample_questions
        })

    prompt = f"""
You are selecting the most relevant lessons from a Siddha medicine knowledge tree.

User query:
{query}

Lesson index:
{compact_json(lesson_index)}

Return ONLY valid JSON.

Format:
{{
  "thinking": "short reason",
  "lesson_ids": [1, 2]
}}

Rules:
- Select 1 to 2 lesson_ids.
- Prefer the single most specific lesson when the query clearly belongs to one lesson.
- Choose lessons most likely to contain the answer.
- Do not create new lesson IDs.
- Keep thinking short.
"""

    result = groq_json(prompt, model=model, max_tokens=300)

    lesson_ids = result.get("lesson_ids", [])

    valid_ids = {lesson["lesson_id"] for lesson in tree["lessons"]}
    lesson_ids = [x for x in lesson_ids if x in valid_ids]

    return {
        "thinking": result.get("thinking", ""),
        "lesson_ids": lesson_ids
    }


In [66]:
def llm_select_items(query, tree, lesson_ids, model="llama-3.3-70b-versatile"):
    candidates = []

    for lesson in tree["lessons"]:
        if lesson["lesson_id"] in lesson_ids:
            for item in lesson["items"]:
                candidates.append({
                    "node_id": f"Q{item['id']}",
                    "id": item["id"],
                    "lesson_name": lesson["lesson_name"],
                    "question": item["question"]
                })

    if not candidates:
        return {
            "thinking": "No candidate lessons found.",
            "node_list": []
        }

    prompt = f"""
You are selecting relevant Q&A nodes from a Siddha medicine knowledge tree.

User query:
{query}

Candidate Q&A nodes:
{compact_json(candidates)}

Return ONLY valid JSON.

Format:
{{
  "thinking": "short reason",
  "node_list": ["Q10", "Q25"]
}}

Rules:
- Select 1 to 8 node_ids.
- Choose questions that directly match the user query.
- Do not create node_ids.
- If no relevant node exists, return an empty list.
- Keep thinking short.
"""

    result = groq_json(prompt, model=model, max_tokens=400)

    valid_node_ids = {item["node_id"] for item in candidates}
    node_list = result.get("node_list", [])
    node_list = [x for x in node_list if x in valid_node_ids]

    return {
        "thinking": result.get("thinking", ""),
        "node_list": node_list
    }


In [67]:
def find_nodes_by_ids(tree, target_node_ids):
    found = []

    target_node_ids = set(target_node_ids)

    for lesson in tree["lessons"]:
        for item in lesson["items"]:
            node_id = f"Q{item['id']}"

            if node_id in target_node_ids:
                found.append({
                    "node_id": node_id,
                    "lesson_id": lesson["lesson_id"],
                    "lesson_name": lesson["lesson_name"],
                    "id": item["id"],
                    "title": item["question"],
                    "text": item["answer"]
                })

    return found


In [68]:
def generate_answer(query, nodes, model="llama-3.3-70b-versatile"):
    if not nodes:
        return "இந்த கேள்விக்கு பொருத்தமான தகவல் இந்த knowledge tree-ல் கிடைக்கவில்லை."

    context_parts = []

    for node in nodes:
        context_parts.append(
            f"[Node: {node['node_id']} | Lesson: {node['lesson_name']}]\n"
            f"Question: {node['title']}\n"
            f"Answer: {node['text']}"
        )

    context = "\n\n---\n\n".join(context_parts)

    prompt = f"""
You are a Siddha medicine assistant.

Answer the user using ONLY the provided context.

User query:
{query}

Context:
{context}

Rules:
- Answer in Tamil.
- Use only the provided context.
- Do not invent remedies.
- Mention the matching source node IDs.
- Add a short safety note recommending a qualified doctor/practitioner for serious symptoms.

Answer:
"""

    response = groq_client.chat.completions.create(
        model=model,
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0,
        max_completion_tokens=700
    )

    return response.choices[0].message.content.strip()


In [69]:
def vectorless_rag(query, tree, model="llama-3.3-70b-versatile", verbose=True):
    if verbose:
        print("=" * 60)
        print("Query:", query)
        print("=" * 60)

    lesson_search = llm_select_lessons(query, tree, model=model)
    lesson_ids = lesson_search["lesson_ids"]

    if verbose:
        print("Lesson reasoning:", lesson_search["thinking"])
        print("Selected lesson IDs:", lesson_ids)

    item_search = llm_select_items(query, tree, lesson_ids, model=model)
    node_ids = item_search["node_list"]

    if verbose:
        print("Item reasoning:", item_search["thinking"])
        print("Selected node IDs:", node_ids)

    nodes = find_nodes_by_ids(tree, node_ids)

    if verbose:
        print("Retrieved nodes:", [n["title"] for n in nodes])

    answer = generate_answer(query, nodes, model=model)

    if verbose:
        print()
        print("Answer:")
        print(answer)

    return {
        "query": query,
        "lesson_ids": lesson_ids,
        "node_ids": node_ids,
        "nodes": nodes,
        "answer": answer
    }


In [70]:
result = vectorless_rag(
    query="மலச்சிக்கல் தீர என்ன செய்யலாம்?",
    tree=tree
)


Query: மலச்சிக்கல் தீர என்ன செய்யலாம்?
Lesson reasoning: Digestive health issues
Selected lesson IDs: [7]
Item reasoning: Direct match for மலச்சிக்கல் தீர
Selected node IDs: ['Q284', 'Q285', 'Q286', 'Q287', 'Q290', 'Q291', 'Q293', 'Q294']
Retrieved nodes: ['மலச்சிக்கல் தீர', 'மலச்சிக்கல் நீங்கி மூலம் கட்டுப்பட', 'மலச்சிக்கல் குணமாக', 'மலச்சிக்கல் நீங்க', 'மலச்சிக்கல் தீர', 'மலக்கட்டு தீர', 'மலச்சிக்கலை போக்க', 'மலம் கழித்தல் கிராணி']

Answer:
மலச்சிக்கல் தீர அகத்திகீரை வாரம் 1 நாள் சமைத்து உண்ண வேண்டும் (Node: Q284). பப்பாளிபழம் தினந்தோறும் சாப்பிட்டு வர குணமாகும் (Node: Q285). பார்லி அரிசி 20கிராம், புளிய இலை 40 கிராம் காய்ச்சி கஷாயமாக்கி குடித்து வரலாம் (Node: Q286). முளைக்கீரை சாப்பிட்டு வரலாம் (Node: Q287). இரவில் மாம்பழம் சாப்பிடலாம் (Node: Q290). முள்ளங்கி இலைசாறு 5மி.லி.3 வேளை சாப்பிட்டு வரலாம் (Node: Q291). நார்த்தங்காய் ஊறுகாய் சாப்பிடவும் (Node: Q293). பாதாளமூலி சதையை சிறுதுண்டுகளாக்கி மிளகுதூள் சேர்த்து சாப்பிட தீரும் (Node: Q294).

குறிப்பு: தீவிர அறிகுறிகள் இருந்தால், தகுத

In [71]:
result = vectorless_rag(
    query="இருமல் குணமாக என்ன மருந்து?",
    tree=tree
)


Query: இருமல் குணமாக என்ன மருந்து?
Lesson reasoning: Cough is related to respiratory health
Selected lesson IDs: [8]
Item reasoning: Direct match for cough cure
Selected node IDs: ['Q140', 'Q157', 'Q158', 'Q762']
Retrieved nodes: ['இருமல் குணமாக', 'இருமல் குணமாக', 'கக்குவான் இருமல் வேகம் குறைய', 'இருமல் உடனே நிற்க']

Answer:
இருமல் குணமாக வெந்தயக்கீரை சமைத்து சாப்பிட்டு வரலாம் (Node: Q140). மாதுளம்பூ பொடியுடன் பனங்கற்கண்டு சேர்த்து காலை, மாலை 1 தேக்கரண்டி சாப்பிட்டு வரலாம் (Node: Q157). முற்றிய வெண்டைக்காயைச் சூப் செய்து குடித்து வந்தால் குணமாகும் (Node: Q762). 

குறிப்பு: தீவிரமான இருமல் அறிகுறிகள் இருந்தால், தகுதிவாய்ந்த மருத்துவர்/சித்த மருத்துவரின் ஆலோசனையை நாடுங்கள்.
